# Gold: NYC Taxi Daily Trip Summary (v1)

Reads the `nyc_taxi.yellow_tripdata` bronze table (see
`data-generators/bronze_ingest_nyc_taxi.ipynb`) and aggregates it into a
per-pickup-date summary: trip count, total revenue, average distance, and
average fare. Persists the result as a managed Spark table for demo
dashboards/queries.

Re-runnable: `mode("overwrite")`, so re-running just refreshes the table.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

## Configuration

In [ ]:
DATABASE_NAME = "nyc_taxi"
SOURCE_TABLE = "yellow_tripdata"
TARGET_TABLE = "gold_taxi_trip_summary_v1"

## Load bronze table

In [ ]:
df = spark.table(f"{DATABASE_NAME}.{SOURCE_TABLE}")

## Transform: daily trip summary

Aggregates by pickup date: trip count, total revenue (`total_amount`),
average trip distance, and average fare.

In [ ]:
summary_df = (
    df
    .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .groupBy("pickup_date")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
    )
    .orderBy("pickup_date")
)

## Persist as a managed Spark table

In [ ]:
summary_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TARGET_TABLE}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_days FROM {DATABASE_NAME}.{TARGET_TABLE}").show()
spark.sql(f"SELECT * FROM {DATABASE_NAME}.{TARGET_TABLE} ORDER BY pickup_date LIMIT 10").show()